# Cheyenne River Corridor — Study Area Definition

Defines the area of interest (AOI) for the cottonwood classification project: the Cheyenne River and its **direct tributaries** from **Angostura Reservoir** (near Hot Springs, SD) downstream to **Oahe Reservoir** (Missouri River, near Eagle Butte, SD).

**Outputs** saved to `../data/cheyenne_corridor_aoi.gpkg`:
- `study_area` — AOI polygon (main stem + 10 km buffer)
- `flowlines` — NHD stream reaches in the corridor
- `usgs_gauges` — USGS gauge sites along the corridor

A hydrograph section at the end fetches daily discharge for all corridor gauges (1990–2024).

In [ ]:
import os, sys
from pathlib import Path

# --- PROJ/GDAL data paths: must be set BEFORE any geospatial import ---
# The Jupyter kernel starts without `conda activate`, so these are otherwise unset.
# sys.prefix alone is not enough: in a venv layered on a conda env (the CyVerse
# HYR-SENSE overlay), sys.prefix is the venv, which has no share/proj — the data
# lives under sys.base_prefix. Check both and use whichever actually exists.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)


def _resolve_data_dir():
    """Locate the project data directory.

    Priority:
      1. $VBET_DATA_DIR, if set — explicit override for keeping big rasters
         somewhere other than the repo.
      2. <repo root>/data, found by walking up from the working directory.
         cwd-independent, so it works from notebooks/, from the repo root,
         and under papermill.
      3. ../data, relative to the working directory.

    On CyVerse the repo is cloned into ~/data-store, so <repo root>/data is
    ALREADY persistent storage. Redirecting elsewhere is unnecessary and was
    previously the cause of "cheyenne_corridor_aoi.gpkg not found" — the
    kernel pointed at an empty directory while the data sat in the repo.
    """
    env = os.environ.get("VBET_DATA_DIR")
    if env:
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p / "data"
    return Path("../../data")

DATA_DIR = _resolve_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import folium
from shapely.geometry import box
from shapely.ops import unary_union
from pynhd import NLDI

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO = _repo_root()

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

FIG_SUBDIR = "study_area"

print("Ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Corridor anchor gauges — confirmed USGS stations on the Cheyenne River
#   UPSTREAM   = just below Angostura Dam (western / headwater end of corridor)
#   REFERENCE  = Red Shirt (center; well-tested with NLDI)
#   DOWNSTREAM = near Eagle Butte (eastern / Oahe Reservoir end of corridor)
# ---------------------------------------------------------------------------
UPSTREAM_GAUGE   = "06401500"   # Cheyenne R below Angostura Dam, SD
REFERENCE_GAUGE  = "06403700"   # Cheyenne R at Red Shirt, SD
DOWNSTREAM_GAUGE = "06439500"   # Cheyenne R near Eagle Butte, SD

nldi = NLDI()

stn_upstream   = nldi.getfeature_byid("nwissite", f"USGS-{UPSTREAM_GAUGE}").to_crs(4326)
stn_reference  = nldi.getfeature_byid("nwissite", f"USGS-{REFERENCE_GAUGE}").to_crs(4326)
stn_downstream = nldi.getfeature_byid("nwissite", f"USGS-{DOWNSTREAM_GAUGE}").to_crs(4326)

for label, stn in [("Upstream  (Angostura Dam)", stn_upstream),
                   ("Reference (Red Shirt)",     stn_reference),
                   ("Downstream (Eagle Butte)",  stn_downstream)]:
    pt   = stn.geometry.iloc[0]
    name = stn['name'].iloc[0] if 'name' in stn.columns else stn['identifier'].iloc[0]
    print(f"{label}:\n  {name}\n  lon={pt.x:.4f}  lat={pt.y:.4f}\n")

In [ ]:
# ---------------------------------------------------------------------------
# Bounding box for the corridor  (anchor locations + 0.5° padding ~55 km)
# ---------------------------------------------------------------------------
pt_up = stn_upstream.geometry.iloc[0]
pt_dn = stn_downstream.geometry.iloc[0]

BBOX_BUFFER = 0.5
corridor_bbox = gpd.GeoDataFrame(
    geometry=[box(
        min(pt_up.x, pt_dn.x) - BBOX_BUFFER,
        min(pt_up.y, pt_dn.y) - BBOX_BUFFER,
        max(pt_up.x, pt_dn.x) + BBOX_BUFFER,
        max(pt_up.y, pt_dn.y) + BBOX_BUFFER,
    )],
    crs=4326,
)

# ---------------------------------------------------------------------------
# Main stem — navigate upstream from Eagle Butte to Angostura
# ---------------------------------------------------------------------------
print("Fetching main stem flowlines … (~30 s)")
flw_main = nldi.navigate_byid(
    fsource="nwissite",
    fid=f"USGS-{DOWNSTREAM_GAUGE}",
    navigation="upstreamMain",
    source="flowlines",
    distance=600,
).clip(corridor_bbox)

# Flowlines use 'comid' as the reach identifier (not 'identifier', which is for gauges)
_id_col = "comid" if "comid" in flw_main.columns else flw_main.columns[0]
print(f"  Flowline columns: {list(flw_main.columns)}")

# ---------------------------------------------------------------------------
# Tributaries — navigate all upstream tributaries from Eagle Butte, then
# keep only reaches within 30 km of the Cheyenne main stem.
# This excludes the Belle Fourche headwaters (which extend far north/west
# of the Cheyenne stem) while retaining direct-drainage tributaries.
# ---------------------------------------------------------------------------
print("Fetching tributary flowlines … (~60 s)")
flw_tribs_raw = nldi.navigate_byid(
    fsource="nwissite",
    fid=f"USGS-{DOWNSTREAM_GAUGE}",
    navigation="upstreamTributaries",
    source="flowlines",
    distance=600,
).clip(corridor_bbox)

TRIB_BUFFER_DEG = 0.27  # ~30 km at lat 44°N
main_buffer = unary_union(flw_main.geometry.values).buffer(TRIB_BUFFER_DEG)
flw_tribs = flw_tribs_raw[flw_tribs_raw.geometry.intersects(main_buffer)].copy()

# ---------------------------------------------------------------------------
# Combine and deduplicate on comid (NHD reach identifier for flowlines)
# ---------------------------------------------------------------------------
flw_corridor = gpd.GeoDataFrame(
    pd.concat([flw_main, flw_tribs], ignore_index=True),
    crs=4326,
).drop_duplicates(subset=_id_col)

print(f"\nMain stem reaches:    {len(flw_main)}")
print(f"Direct trib reaches:  {len(flw_tribs)}")
print(f"Total unique reaches: {len(flw_corridor)}")

In [ ]:
# ---------------------------------------------------------------------------
# USGS gauge sites within the corridor
# ---------------------------------------------------------------------------
print("Fetching USGS gauge sites …")
gauges_raw = nldi.navigate_byid(
    fsource="nwissite",
    fid=f"USGS-{DOWNSTREAM_GAUGE}",
    navigation="upstreamTributaries",
    source="nwissite",
    distance=600,
).clip(corridor_bbox)

# Keep gauges within 30 km of the Cheyenne main stem
gauges_on_corridor = gauges_raw[gauges_raw.geometry.intersects(main_buffer)].copy()

# Add anchor stations (navigate_byid excludes the starting feature)
anchor_stns = gpd.GeoDataFrame(
    pd.concat([stn_upstream, stn_reference, stn_downstream], ignore_index=True),
    crs=4326,
)[["identifier", "geometry"]]

gauges_corridor = gpd.GeoDataFrame(
    pd.concat(
        [anchor_stns, gauges_on_corridor[["identifier", "geometry"]]],
        ignore_index=True,
    ),
    crs=4326,
).drop_duplicates(subset="identifier")

# Display sorted upstream → downstream (west → east)
gauges_corridor["_lon"] = gauges_corridor.geometry.x
gauges_sorted = gauges_corridor.sort_values("_lon").drop(columns="_lon")
print(f"\n{len(gauges_corridor)} USGS gauge sites in the Cheyenne River corridor:\n")
for _, row in gauges_sorted.iterrows():
    print(f"  {row['identifier']}   lon={row.geometry.x:.3f}  lat={row.geometry.y:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# Export corridor layers to GeoPackage
# ---------------------------------------------------------------------------
aoi_geom = unary_union(flw_main.geometry.values).buffer(0.09)  # ~10 km buffer
aoi_gdf  = gpd.GeoDataFrame(
    {"description": ["Cheyenne River corridor — Angostura Dam to Oahe Reservoir"]},
    geometry=[aoi_geom],
    crs=4326,
)

OUTPUT_GPKG = str(DATA_DIR / "cheyenne_corridor_aoi.gpkg")

aoi_gdf.to_file(OUTPUT_GPKG, driver="GPKG", layer="study_area")
flw_corridor.to_file(OUTPUT_GPKG, driver="GPKG", layer="flowlines")
gauges_corridor.to_file(OUTPUT_GPKG, driver="GPKG", layer="usgs_gauges")

print(f"Saved → {OUTPUT_GPKG}")
print(f"  study_area  : {len(aoi_gdf)} polygon")
print(f"  flowlines   : {len(flw_corridor)} stream reaches")
print(f"  usgs_gauges : {len(gauges_corridor)} gauge sites")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive Folium map of the study corridor
# ---------------------------------------------------------------------------
ANCHOR_IDS = {UPSTREAM_GAUGE, REFERENCE_GAUGE, DOWNSTREAM_GAUGE}
center_lat = (pt_up.y + pt_dn.y) / 2
center_lon = (pt_up.x + pt_dn.x) / 2

# CARTO now requires an API key for their tile servers, so folium's built-in
# "CartoDB positron" renders with an "API KEY REQUIRED" watermark. Esri's light
# grey canvas looks the same and needs no key. Same pattern as notebook 01.
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services"

m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles=None)
folium.TileLayer(
    tiles=f"{ESRI}/Canvas/World_Light_Gray_Base/MapServer/tile/{{z}}/{{y}}/{{x}}",
    attr="Esri", name="Light grey canvas",
).add_to(m)
folium.TileLayer(
    tiles=f"{ESRI}/World_Imagery/MapServer/tile/{{z}}/{{y}}/{{x}}",
    attr="Esri", name="Aerial imagery", show=False,
).add_to(m)

folium.GeoJson(
    aoi_gdf, name="AOI corridor",
    style_function=lambda x: {"color": "orange", "weight": 2, "fillOpacity": 0.07},
).add_to(m)

folium.GeoJson(
    flw_main, name="Cheyenne River (main stem)",
    style_function=lambda x: {"color": "#1a4fa8", "weight": 3},
).add_to(m)

folium.GeoJson(
    flw_tribs, name="Direct tributaries",
    style_function=lambda x: {"color": "#5599e0", "weight": 1.2},
).add_to(m)

for _, row in gauges_corridor.iterrows():
    site_id = row["identifier"].replace("USGS-", "")
    pt = row.geometry
    is_anchor = site_id in ANCHOR_IDS
    folium.CircleMarker(
        location=[pt.y, pt.x],       # .iloc[0] not needed — iterrows() gives scalar Series
        radius=8 if is_anchor else 5,
        color="red" if is_anchor else "#e8a100",
        fill=True,
        fill_opacity=0.9,
        popup=folium.Popup(f"<b>{row['identifier']}</b>", max_width=180),
        tooltip=row["identifier"],
    ).add_to(m)

folium.LayerControl().add_to(m)
m

## Hydrograph — Daily Discharge for Corridor Gauges

Cottonwood recruitment depends on **when** water arrives, not just how much: seedlings establish
on bars exposed by a spring peak that then recedes slowly enough for roots to follow the water
table. So the useful picture is a **typical year** — flow by day of year — rather than a
year-to-year series.

We fetch the full period of record for each gauge, because record lengths differ enormously
along this corridor and that turns out to matter.

In [ ]:
import warnings
try:
    import dataretrieval.nwis as nwis
except ImportError:
    raise ImportError("Run: pip install dataretrieval")

# The seven Cheyenne main-stem gauges, west to east. The corridor layer also contains
# ~50 other surface-water gauges on tributaries (and ~200 groundwater sites with no
# discharge at all); widen this list if you want them.
MAIN_STEM = {
    "06401500": "below Angostura Dam",
    "06402600": "near Buffalo Gap",
    "06403700": "at Red Shirt",
    "06408650": "near Scenic",
    "06423500": "near Wasta",
    "06438500": "near Plainview",
    "06439500": "near Eagle Butte",
}

hydrographs = {}
for site_id in MAIN_STEM:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df, _ = nwis.get_dv(sites=[site_id], parameterCd="00060",
                            start="1900-01-01", end="2024-12-31")   # full period of record
    q_col = next((c for c in df.columns if "00060" in c and not c.endswith("_cd")), None)
    if df is not None and len(df) and q_col:
        s = pd.to_numeric(df[q_col], errors="coerce").dropna()
        s.index = pd.to_datetime(s.index)
        hydrographs[site_id] = s.rename(site_id)

df_q = pd.DataFrame(hydrographs).sort_index()

# Record summary. These gauges are far less comparable than a list of seven suggests.
record = pd.DataFrame({
    "name":  [MAIN_STEM[g] for g in df_q.columns],
    "first": [df_q[g].dropna().index.min().year for g in df_q.columns],
    "last":  [df_q[g].dropna().index.max().year for g in df_q.columns],
    "years": [df_q[g].dropna().index.year.nunique() for g in df_q.columns],
    "days":  [int(df_q[g].notna().sum()) for g in df_q.columns],
}, index=df_q.columns)
print(record.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Typical-year hydrograph: flow by day of year
# ---------------------------------------------------------------------------
# Rather than demanding whole calendar years, we require a minimum number of years
# contributing to each day of year. That matters here: Angostura reports only ~181
# days a year (it is operated seasonally), so a "complete year" rule would throw away
# a continuous record going back to 1945. It also keeps a gauge with only two years
# of data from being drawn as if it described a typical year.
MIN_YEARS_PER_DOY = 10
MODERN_SINCE      = 2010        # gauges last reporting before this are labelled historical
SEED_RELEASE      = (145, 180)  # approx. day-of-year window for plains cottonwood seed
                                # release (late May - late June). Verify for this river.
REF_YEAR = 2001                 # any non-leap year, to turn day-of-year back into dates

def doy_stats(gid):
    """Flow percentiles by day of year, using only days backed by enough years."""
    s = df_q[gid].dropna()
    grp = s.groupby(s.index.dayofyear)
    st = pd.DataFrame({q: grp.quantile(v) for q, v in
                       [("p10", .10), ("p25", .25), ("p50", .50), ("p75", .75), ("p90", .90)]})
    st["n_years"] = grp.apply(lambda x: x.index.year.nunique())
    st = st[st.n_years >= MIN_YEARS_PER_DOY]
    st = st[st.index <= 365]     # day 366 would wrap onto the next REF_YEAR
    if st.empty:
        return st
    st = st.rolling(7, center=True, min_periods=1).mean()
    st.index = pd.to_datetime([f"{REF_YEAR}-{d:03d}" for d in st.index], format="%Y-%j")
    return st

plotted = {g: doy_stats(g) for g in df_q.columns}
plotted = {g: st for g, st in plotted.items() if not st.empty}

fig, axes = plt.subplots(len(plotted), 1, figsize=(6.5, 1.4 * len(plotted)),
                         sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, (gid, st) in zip(axes, plotted.items()):
    hist = record.loc[gid, "last"] < MODERN_SINCE
    col  = "#8d6e63" if hist else "#1a3f7a"
    ax.fill_between(st.index, st.p10, st.p90, color=col, alpha=0.15, linewidth=0)
    ax.fill_between(st.index, st.p25, st.p75, color=col, alpha=0.35, linewidth=0)
    ax.plot(st.index, st.p50, color=col, linewidth=1.8)

    # The window that matters for recruitment: seedlings need a peak that recedes slowly.
    ax.axvspan(pd.Timestamp(f"{REF_YEAR}-01-01") + pd.Timedelta(days=SEED_RELEASE[0] - 1),
               pd.Timestamp(f"{REF_YEAR}-01-01") + pd.Timedelta(days=SEED_RELEASE[1] - 1),
               color="#2E7D32", alpha=0.10, zorder=0)

    ax.set_yscale("log")   # daily flow spans orders of magnitude
    ax.set_ylabel("cfs")
    tag = "  — historic (no modern record)" if hist else ""
    ax.set_title(f"{gid}  {MAIN_STEM[gid]}  ({record.loc[gid,'first']}\u2013"
                 f"{record.loc[gid,'last']}, {record.loc[gid,'years']} yrs){tag}",
                 fontsize=8, loc="left", color="#6d4c41" if hist else "black")
    ax.grid(True, alpha=0.25, which="both")

axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.suptitle("Cheyenne River main stem — typical year by day of year\n"
             "line = median, bands = 25\u201375th and 10\u201390th percentiles; "
             "green = cottonwood seed release", fontsize=11)
savefig("hydrograph_by_gauge")
plt.show()

dropped = [g for g in df_q.columns if g not in plotted]
if dropped:
    print("Not drawn (fewer than "
          f"{MIN_YEARS_PER_DOY} years on any day): {', '.join(dropped)}")

In [ ]:
# ---------------------------------------------------------------------------
# Record availability — the reason those panels are not comparable
# ---------------------------------------------------------------------------
days_per_year = df_q.notna().groupby(df_q.index.year).sum()
years = np.arange(int(record["first"].min()), int(record["last"].max()) + 1)
order = record.sort_values("first").index

fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(order) + 1.8))
for row, gid in enumerate(order):
    n = days_per_year[gid].reindex(years, fill_value=0)
    full, part = n >= 300, (n > 0) & (n < 300)
    ax.scatter(years[full.values], [row] * full.sum(), marker="s", s=22,
               color="#1a3f7a", label="full year" if row == 0 else None)
    ax.scatter(years[part.values], [row] * part.sum(), marker="s", s=22,
               color="#90a4ae", label="partial / seasonal" if row == 0 else None)

ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{g}  {MAIN_STEM[g]}" for g in order], fontsize=9)
ax.set_xlabel("Year")
ax.set_title("Discharge record by main-stem gauge", fontsize=11, loc="left")
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax.grid(True, axis="x", alpha=0.25)
plt.tight_layout()
savefig("record_availability")
plt.show()

print("Full daily record for all seven gauges is kept in `df_q` for later analysis.")

### Corridor envelope — all gauges pooled

The panels above sit on very different periods of record, and the plot beneath them shows why.
This last view does not try to reconcile them. It pools every main-stem daily value by day of
year and treats the corridor as a single unit, the way a basin-wide snow plot does.

The bands therefore mix between-gauge and between-year spread, so read this as *what a day in
June looks like on the Cheyenne main stem* — not as a comparison between sites.

Eagle Butte is drawn separately rather than folded into the envelope. It is the only long record
at the downstream end of the corridor, and the only one reaching back before Angostura Dam
(completed ~1949 — unverified, see plan §17); its 1934–1967 record straddles that date rather
than predating it, so it is **not** a clean pre-regulation baseline.

In [ ]:
# ---------------------------------------------------------------------------
# Corridor-wide typical year — all main-stem gauges pooled
# ---------------------------------------------------------------------------
# Deliberately NOT period-matched: no window covers all seven gauges (Buffalo Gap
# starts the year Eagle Butte stops), so pooling by day of year and describing the
# corridor as one unit is the honest version of "combine the stations".
POOL_GAUGES = list(df_q.columns)      # drop "06439500" here for a post-regulation view

pool = (df_q[POOL_GAUGES]
        .melt(ignore_index=False, var_name="site", value_name="cfs")
        .dropna(subset=["cfs"]))
pool["doy"] = pool.index.dayofyear
pool = pool[pool.doy <= 365]          # as above: day 366 would wrap onto the next year

grp = pool.groupby("doy")["cfs"]
env = pd.DataFrame({q: grp.quantile(v) for q, v in
                    [("p10", .10), ("p25", .25), ("p50", .50), ("p75", .75), ("p90", .90)]})
env = env.rolling(7, center=True, min_periods=1).mean()
env.index = pd.to_datetime([f"{REF_YEAR}-{d:03d}" for d in env.index], format="%Y-%j")

fig, ax = plt.subplots(figsize=(7.5, 4.6), constrained_layout=True)
ax.fill_between(env.index, env.p10, env.p90, color="#1a3f7a", alpha=0.15, linewidth=0,
                label="10\u201390th percentile")
ax.fill_between(env.index, env.p25, env.p75, color="#1a3f7a", alpha=0.35, linewidth=0,
                label="25\u201375th percentile")
ax.plot(env.index, env.p50, color="#1a3f7a", linewidth=2.0, label="median")

# The historic gauge gets its own line rather than disappearing into the envelope:
# it is the only long record at the downstream end of the corridor.
if "06439500" in POOL_GAUGES:
    eb = doy_stats("06439500")
    if not eb.empty:
        ax.plot(eb.index, eb.p50, color="#8d6e63", linewidth=1.8, linestyle="--",
                label="historic (Eagle Butte)")

    # Its variability, kept light: one whisker per month rather than a third filled band,
    # which would collide with the two blue ones. Quartiles, NOT mean +/- SD -- daily flow
    # is strongly right-skewed, so a symmetric SD bar runs negative and cannot be drawn on
    # a log axis. The 2007-08 reactivation is ~6% of the record and the quartiles are
    # robust to its 65,200 cfs flood; the extreme tail would not be.
    eb_daily = df_q["06439500"].dropna()
    eb_m     = eb_daily.groupby(eb_daily.index.month)
    eb_box   = pd.DataFrame({"p25": eb_m.quantile(.25), "p50": eb_m.quantile(.50),
                             "p75": eb_m.quantile(.75)})
    xm = [pd.Timestamp(f"{REF_YEAR}-{m:02d}-15") for m in eb_box.index]
    ax.errorbar(xm, eb_box.p50,
                yerr=[eb_box.p50 - eb_box.p25, eb_box.p75 - eb_box.p50],
                fmt="o", ms=4, color="#8d6e63", ecolor="#8d6e63", elinewidth=1.1,
                capsize=3, alpha=0.9, zorder=5,
                label="Eagle Butte 25\u201375th (monthly)")

ax.axvspan(pd.Timestamp(f"{REF_YEAR}-01-01") + pd.Timedelta(days=SEED_RELEASE[0] - 1),
           pd.Timestamp(f"{REF_YEAR}-01-01") + pd.Timedelta(days=SEED_RELEASE[1] - 1),
           color="#2E7D32", alpha=0.10, zorder=0, label="cottonwood seed release")

ax.set_yscale("log")                  # daily flow spans orders of magnitude
ax.set_ylabel("cfs")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(True, alpha=0.25, which="both")
# Six entries and a seasonal hump leave no free corner — put the key below the axes
# so it cannot land on the data whatever the record turns out to look like.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3,
          fontsize=9, frameon=False)
ax.set_title(f"Cheyenne River main stem — corridor envelope, "
             f"{len(POOL_GAUGES)} gauges pooled\n"
             f"{len(pool):,} gauge-days, full period of record for each gauge",
             fontsize=11, loc="left")
savefig("corridor_envelope")
plt.show()